In [0]:
%pip install pendulum

In [0]:
import json
from pathlib import Path
import pendulum
import requests

In [0]:
#Detecta el catálogo(ubicación) actual (suele ser 'workspace' o 'main')
current_catalog = spark.sql("select current_catalog()").first()[0]
catalog = current_catalog
schema = "raw"
volume = "tvmze"

#Crear esquema y volumen si no existen
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.{volume}")

In [0]:
#Elimina todos loa archivos y subcarpetas dentro de la ruta
dbutils.fs.rm("/Volumes/workspace/raw/tvmze", recurse = True)

In [0]:
#Crear widgets
dbutils.widgets.text("start_year", "2018", "Start Year (YYYY-MM-DD)")
dbutils.widgets.text("end_year", "", "End Year (YYYY-MM-DD)")
dbutils.widgets.text("departament", "", "Departament")
dbutils.widgets.text("municipality", "", "Municipality")
dbutils.widgets.text("tipo_zona_predio", "urbano", "Tipo de Zona")
dbutils.widgets.text("dinamica_inmobiliaria", "", "Dinamica Inmobiliaria")
dbutils.widgets.text("output_dir", "/Volumes/workspace/raw/tvmze", "Output Directory")
dbutils.widgets.text("api_endpoint", "https://www.datos.gov.co/resource/7y2j-43cv.json", "API endpoint")
dbutils.widgets.text("timeout", "30", "Time Out")

#Leer valores de los widgets
start_year = dbutils.widgets.get("start_year")
end_year = dbutils.widgets.get("end_year")
departament = dbutils.widgets.get("departament")
municipality = dbutils.widgets.get("municipality")
tipo_zona_predio = dbutils.widgets.get("tipo_zona_predio")
dinamica_inmobiliaria = dbutils.widgets.get("dinamica_inmobiliaria")
output_dir = dbutils.widgets.get("output_dir")
api_endpoint = dbutils.widgets.get("api_endpoint")
API_URL = api_endpoint
if end_year == "":
    end_year = start_year
for year in range(int(start_year), int(end_year) + 1):
    params = {
        "$where": f"year_radica='{year}'",
        "$limit": 10000
    }
    r = requests.get(api_endpoint, params=params, timeout=30)
timeout = int(dbutils.widgets.get("timeout"))

In [0]:
saved_files = []
current_year = int(start_year)
end_year = int(end_year)

session = requests.Session()

while current_year <= end_year:

    params = {
        "$where": f"year_radica='{current_year}'",
        "$limit": 10000
    }
    year_output_dir = Path(output_dir) / f"{current_year}"
    year_output_dir.mkdir(parents=True, exist_ok=True)

    file_path = year_output_dir / "tvmaze.json"

    try:
        response = session.get(API_URL, params=params, timeout=timeout)
        response.raise_for_status()
    except requests.RequestException:
        current_year += 1
        continue

    file_path.write_bytes(response.content)

    saved_files.append(str(file_path))
    current_year += 1
print(f"Archivos guardados: {len(saved_files)}")